In [1]:
from datetime import datetime
from pathlib import Path
import sys

sys.path.insert(0, str(Path("../..").resolve()))

import ad_safe_lib as ad_safe

print("challenge:", ad_safe.CHALLENGE_DIR)
print("device:", ad_safe.DEVICE)
print("dataset:", ad_safe.DATA_DIR)
examples_root = ad_safe.AD_SAFE_RUNS_DIR / "notebook_examples"
examples_root.mkdir(parents=True, exist_ok=True)


challenge: /home/madmazoku/project/ml-bootcamp-2026/challenge
device: cuda
dataset: /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset


# Tiny Sweep And Metrics Comparison

Build a few jobs programmatically, run the shared training runner, then compare models with the shared evaluation runner.


In [4]:
run_id = "notebook-04-" + datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
output_dir = examples_root / run_id

# Slightly larger subsets reduce variance while keeping notebook runtime manageable.
train_source = ad_safe.DatasetSourceSpec("train", fraction=0.08, seed=40)
eval_sources = {
    "val": ad_safe.DatasetSourceSpec("val", fraction=0.10, seed=101),
    "test": ad_safe.DatasetSourceSpec("test", fraction=0.10, seed=102),
}

# Include one compact pretrained baseline for comparison.
backbones = ["simple_mlp", "simple_cnn", "mobilenet_v3_small"]

backbone_settings = {
    "simple_mlp": {"learning_rate": 8e-4, "epochs": 8, "patience": 3, "batch_size": 16},
    "simple_cnn": {"learning_rate": 4e-4, "epochs": 8, "patience": 3, "batch_size": 16},
    "mobilenet_v3_small": {"learning_rate": 8e-4, "epochs": 3, "patience": 2, "batch_size": 16},
}

jobs = []
for job_index, backbone in enumerate(backbones):
    settings = backbone_settings[backbone]

    # Scratch simple backbones must train end-to-end, not classifier-only.
    train_all_layers = backbone.startswith("simple_")

    phase = ad_safe.PhaseSpec(
        job_index=job_index,
        phase_index=0,
        prefix=backbone,
        name="quick",
        requested_seed=40 + job_index,
        config=ad_safe.TrainingConfig(
            base_model=backbone,
            epochs=settings["epochs"],
            patience=settings["patience"],
            batch_size=settings["batch_size"],
            learning_rate=(settings["learning_rate"],),
            resplit_runs=1,
        ),
        unfreeze_all=train_all_layers,
        model_filename=f"{backbone}.pt",
        history_filename=f"{backbone}_history.png",
        json_filename=f"{backbone}.json",
    )
    jobs.append(
        ad_safe.JobSpec(
            job_index=job_index,
            job_id=backbone,
            display_name=backbone,
            backbone=backbone,
            phases=(phase,),
        )
    )

plan = ad_safe.RunPlan(
    output_dir=output_dir,
    run_id=run_id,
    train_split="train",
    eval_splits=("val", "test"),
    jobs=tuple(jobs),
    resume=False,
    force=True,
    setup_path=output_dir / "setup.json",
    check_foreign_contract=False,
    train_source=train_source,
    eval_sources=eval_sources,
)

result = ad_safe.run_training_plan(plan)
result.metrics_csv_path


Using device: cuda
Run id: notebook-04-2026-04-22-16-12-05
Output dir: /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-04-2026-04-22-16-12-05
Train split: train
Eval splits: val, test
Resume: False
Force: True
Cooldown: {'every_epochs': 0, 'seconds': 0.0, 'gpu_max_temp': 0, 'gpu_resume_temp': 0, 'gpu_temp_check_seconds': 15.0}
Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/train...
Using stratified dataset source train: 800/10000 samples (fraction=0.08, seed=40)
Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/val...
Using stratified dataset source val: 1000/10000 samples (fraction=0.1, seed=101)
Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/test...
Using stratified dataset source test: 1000/10000 samples (fraction=0.1, seed=102)
Setup saved to /hom

Evaluating (val):   0%|          | 0/5 [00:00<?, ?it/s]

Initial val metrics: acc=0.4500 auc=0.4625 nll=0.6997 conf=0.5251 margin=0.0502 wrong_conf=0.5247 safe_recall=0.3250 unsafe_recall=0.5750


Training:   0%|          | 0/43 [00:00<?, ?it/s]

Evaluating (val):   0%|          | 0/5 [00:00<?, ?it/s]

...

Split 1/1 | Epoch 3/3 | global_epoch=3 | train=0.3485 | val=0.8375 | auc=0.9119 | nll=0.4079 | best=0.8500 @ epoch 2 | comparison=worse | time=2.84s
Loading model from /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-04-2026-04-22-16-12-05/mobilenet_v3_small.pt...
Loaded best model from epoch 2 (score: 0.8500)
Saving model to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-04-2026-04-22-16-12-05/mobilenet_v3_small.pt...
Figure saved to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-04-2026-04-22-16-12-05/mobilenet_v3_small_history.png
Loading model from /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-04-2026-04-22-16-12-05/mobilenet_v3_small.pt...


Evaluating (val):   0%|          | 0/63 [00:00<?, ?it/s]

Evaluating (test):   0%|          | 0/63 [00:00<?, ?it/s]

JSON saved to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-04-2026-04-22-16-12-05/mobilenet_v3_small.json
Metrics CSV saved to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-04-2026-04-22-16-12-05/accuracy.csv
Setup saved to /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-04-2026-04-22-16-12-05/setup.json

Results
prefix              val                                                                                             test                                                                                          
------------------  ----------------------------------------------------------------------------------------------  ----------------------------------------------------------------------------------------------
mobilenet_v3_small  acc=0.7960 auc=0.9057 nll=0.4505 avg_wrong_conf=0.7505 safe_recall=0.6880 unsaf

PosixPath('/home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-04-2026-04-22-16-12-05/accuracy.csv')

In [5]:
ad_safe.run_evaluation_plan(
    ad_safe.EvaluationPlan(
        models=tuple(
            ad_safe.ModelEvalSpec(path=phase.model_path, row_id=phase.prefix)
            for phase in result.phase_results
        ),
        datasets=(
            ad_safe.DatasetEvalSpec(name="val", batch_size=8, source=eval_sources["val"]),
            ad_safe.DatasetEvalSpec(name="test", batch_size=8, source=eval_sources["test"]),
        ),
        output_dir=output_dir,
        sort_key="acc_test",
        title="Sweep comparison",
    )
)


Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/val...
Using stratified dataset source val: 1000/10000 samples (fraction=0.1, seed=101)
Loading dataset from /home/madmazoku/project/ml-bootcamp-2026/challenge/datasets/ml_bootcamp_adsafety_dataset/test...
Using stratified dataset source test: 1000/10000 samples (fraction=0.1, seed=102)

Evaluating simple_mlp.pt...
Loading model from /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-04-2026-04-22-16-12-05/simple_mlp.pt...


Evaluating (val):   0%|          | 0/125 [00:00<?, ?it/s]

val: acc=0.6050 auc=0.6361 nll=1.7450 conf=0.7480 margin=0.4960 wrong_conf=0.7278 safe_recall=0.5900 unsafe_recall=0.6200


Evaluating (test):   0%|          | 0/125 [00:00<?, ?it/s]

test: acc=0.5790 auc=0.5997 nll=2.4280 conf=0.7561 margin=0.5123 wrong_conf=0.7425 safe_recall=0.5480 unsafe_recall=0.6100

Evaluating simple_cnn.pt...
Loading model from /home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-04-2026-04-22-16-12-05/simple_cnn.pt...


...

Evaluating (val):   0%|          | 0/125 [00:00<?, ?it/s]

val: acc=0.7960 auc=0.9057 nll=0.4505 conf=0.8565 margin=0.7130 wrong_conf=0.7505 safe_recall=0.6880 unsafe_recall=0.9040


Evaluating (test):   0%|          | 0/125 [00:00<?, ?it/s]

test: acc=0.7260 auc=0.8557 nll=0.5811 conf=0.8431 margin=0.6861 wrong_conf=0.7618 safe_recall=0.5500 unsafe_recall=0.9020

Sweep comparison
model               val                                                                                             test                                                                                          
------------------  ----------------------------------------------------------------------------------------------  ----------------------------------------------------------------------------------------------
mobilenet_v3_small  acc=0.7960 auc=0.9057 nll=0.4505 avg_wrong_conf=0.7505 safe_recall=0.6880 unsafe_recall=0.9040  acc=0.7260 auc=0.8557 nll=0.5811 avg_wrong_conf=0.7618 safe_recall=0.5500 unsafe_recall=0.9020
simple_cnn          acc=0.6340 auc=0.6905 nll=0.6617 avg_wrong_conf=0.5492 safe_recall=0.6080 unsafe_recall=0.6600  acc=0.6070 auc=0.6527 nll=0.6776 avg_wrong_conf=0.5537 safe_recall=0.5600 unsafe_recall=0.6540
simple_mlp     

EvaluationRunResult(rows=(MetricsMatrixRow(row_id='mobilenet_v3_small', metrics_by_dataset={'val': ClassificationMetrics(accuracy=0.7960000038146973, auc=0.9057279999999999, nll=0.4504724144935608, avg_conf=0.8564935326576233, avg_margin=0.7129870653152466, avg_correct_conf=0.7542790770530701, avg_wrong_conf=0.7505256533622742, safe_recall=0.6880000233650208, unsafe_recall=0.9039999842643738), 'test': ClassificationMetrics(accuracy=0.7260000109672546, auc=0.855664, nll=0.5811190009117126, avg_conf=0.8430588245391846, avg_margin=0.6861175298690796, avg_correct_conf=0.6995909214019775, avg_wrong_conf=0.761802613735199, safe_recall=0.550000011920929, unsafe_recall=0.9020000100135803)}, metadata={'model_name': 'mobilenet_v3_small.pt', 'model_path': '/home/madmazoku/project/ml-bootcamp-2026/challenge/artefacts/ad_safe_runs/notebook_examples/notebook-04-2026-04-22-16-12-05/mobilenet_v3_small.pt', 'rank': 1}), MetricsMatrixRow(row_id='simple_cnn', metrics_by_dataset={'val': ClassificationMetr